In [2]:
!pip install requests kafka-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.3/326.3 kB 4.7 MB/s eta 0:00:00a 0:00:01


In [3]:
from kafka.admin import KafkaAdminClient, NewTopic

admin_client = KafkaAdminClient(bootstrap_servers="host.docker.internal:9093")
topic_list = [NewTopic(name="wiki.recentchange", num_partitions=1, replication_factor=1)]
try:
    admin_client.create_topics(new_topics=topic_list, validate_only=False)
    print("Topic 'wiki.recentchange' created successfully!")
except Exception as e:
    if "TopicAlreadyExists" in str(e):
        print("Topic 'wiki.recentchange' already exists.")
    else:
        print(f"Error creating topic: {e}")
admin_client.close()

Topic 'wiki.recentchange' created successfully!


In [ ]:
import requests
import json
import time 
from kafka import KafkaProducer

producer = KafkaProducer(
    bootstrap_servers="host.docker.internal:9093",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

url = "https://stream.wikimedia.org/v2/stream/recentchange"
headers = {
    "User-Agent": "WikiStreamBot/1.0",
    "Accept": "text/event-stream"
}

print("Starting producer... Press Ctrl+C to stop")
count = 0

try:
    with requests.get(url, headers=headers, stream=True) as resp:
        for line in resp.iter_lines(decode_unicode=True):
            if line.startswith("data: "):
                try:
                    event = json.loads(line[6:])
                    if event.get("meta", {}).get("domain") == "canary":
                        continue

                    clean_event = {
                        "type": event.get("type"),
                        "title": event.get("title"),
                        "user": event.get("user", "Anonymous"),
                        "bot": event.get("bot", False),
                        "comment": event.get("comment", ""),
                        "wiki": event.get("wiki"),
                        "wiki_namespace": event.get("namespace", 0),
                        "timestamp": event.get("meta", {}).get("dt"),
                        "domain": event.get("meta", {}).get("domain"),
                        "server_name": event.get("server_name", ""),
                        "length_old": event.get("length", {}).get("old", 0),
                        "length_new": event.get("length", {}).get("new", 0)
                    }

                    if clean_event["timestamp"] and clean_event["title"]:
                        producer.send("wiki.recentchange", value=clean_event)
                        time.sleep(1)  
                        bot_indicator = "🤖" if clean_event["bot"] else "👤"
                        print(f"{count+1}: {bot_indicator} {clean_event['title']} by {clean_event['user']}")
                        count += 1

                except json.JSONDecodeError:
                    continue

except KeyboardInterrupt:
    print(f"\nStopped after {count} events")
finally:
    producer.close()

Starting producer... Press Ctrl+C to stop
1: 🤖 Category:Information field template with formatting by DPLA bot
2: 🤖 Category:Media contributed by the Digital Public Library of America by DPLA bot
3: 🤖 Category:Media contributed by the National Archives and Records Administration by DPLA bot
4: 🤖 Category:Media contributed by National Archives at College Park - Textual Reference by DPLA bot
5: 👤 ون بيس: البطلات by Mohammdaon
6: 🤖 Category:Taken with Nikon D60 by Rkieferbot
7: 👤 أتجيت (فجوة بركانية) by راشراش
8: 🤖 Category:Digital Public Library of America files missing required SDC statements by DPLA bot
9: 👤 Category:Files from Kremlin.ru, 2026 by Bookish Worm
10: 🤖 Category:Digital Public Library of America files missing creator by DPLA bot
11: 👤 Category:Creative Commons Attribution missing SDC copyright status by Bookish Worm
12: 🤖 Category:PD US by DPLA bot
13: 👤 Category:CC-BY-4.0 by Bookish Worm
14: 👤 File:Mapillary (a3 XsPSEXRWmtreEiLQQhA) (mdroads) 2017-11-25 18H54M36S219.jpg b

In [6]:
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    "wiki.recentchange",
    bootstrap_servers="host.docker.internal:9093",
    auto_offset_reset="earliest",
    value_deserializer=lambda x: json.loads(x.decode("utf-8"))
)

count = 0
for msg in consumer:
    print(f"Received: {msg.value}")
    count += 1
    if count >= 3:
        break
consumer.close()

Received: {'type': 'edit', 'title': 'File:Lenham Post Office.JPG', 'user': 'Rkieferbot', 'bot': True, 'comment': 'Adding [[Category:Taken with Nikon Coolpix L11]] based on EXIF data. [[User_talk:Rkieferbot|Report a bug or suggestion]].', 'wiki': 'commonswiki', 'wiki_namespace': 6, 'timestamp': '2026-02-08T18:29:10.043Z', 'domain': 'commons.wikimedia.org', 'server_name': 'commons.wikimedia.org', 'length_old': 4450, 'length_new': 4492}
Received: {'type': 'categorize', 'title': 'Category:Flickr images reviewed by FlickreviewR 2', 'user': 'FlickreviewR 2', 'bot': True, 'comment': '[[:File:Lichen (5540809950).jpg]] added to category', 'wiki': 'commonswiki', 'wiki_namespace': 14, 'timestamp': '2026-02-08T18:29:10.110Z', 'domain': 'commons.wikimedia.org', 'server_name': 'commons.wikimedia.org', 'length_old': 0, 'length_new': 0}
Received: {'type': 'categorize', 'title': 'Category:Flickr images missing SDC Flickr photo ID', 'user': 'FlickreviewR 2', 'bot': True, 'comment': '[[:File:Lichen (5540